In [1]:
# Install dependencies
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-deps',
    'accelerate==0.34.0',
    'trl==0.9.6',
], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'bitsandbytes==0.45.5',
    'peft==0.11.1',
    'transformers==4.46.0',
    'huggingface_hub==0.23.2',
    'sentencepiece==0.2.0',
    'datasets==2.19.0',
], check=True)

print('Done installing.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00


Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.6 MB/s eta 0:00:00
Done installing.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.9.6 requires tyro>=0.5.11, which is not installed.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
trl 0.9.6 requires numpy<2.0.0,>=1.18.2, but you have numpy 2.0.2 which is incompatible.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.3.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.2 which is incompatible.


In [2]:
#  Verify versions
import accelerate, trl, transformers
print(f'accelerate  : {accelerate.__version__}')   # must be 0.34.0
print(f'trl         : {trl.__version__}')           # must be 0.9.6
print(f'transformers: {transformers.__version__}')  # must be 4.46.0

accelerate  : 0.34.0
trl         : 0.9.6
transformers: 4.46.0


In [3]:
# Imports
import os, json, torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

2026-05-19 04:16:35.470147: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779164195.886892     226 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779164196.008607     226 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779164197.015786     226 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779164197.015829     226 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779164197.015832     226 computation_placer.cc:177] computation placer alr

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


In [4]:
#  Config
HF_USERNAME  = 'kk014'
MODEL_NAME   = 'mistralai/Mistral-7B-v0.1'
REPO_NAME    = f'{HF_USERNAME}/mistral-7b-docstring'
DATA_PATH    = '/kaggle/input/datasets/ascian100/mistral-docstring-data'
OUTPUT_DIR   = '/kaggle/working/output'
MAX_SEQ_LEN  = 512

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

EPOCHS       = 1
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
LR           = 2e-4
WARMUP_RATIO = 0.03

print(f'Repo : {REPO_NAME}')
print(f'Data : {DATA_PATH}')

Repo : kk014/mistral-7b-docstring
Data : /kaggle/input/datasets/ascian100/mistral-docstring-data


In [5]:
#  Login to HuggingFace
from kaggle_secrets import UserSecretsClient
secrets  = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('Logged in to HuggingFace.')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
Logged in to HuggingFace.


In [6]:
#  Load dataset
train_dataset = load_dataset('json',
    data_files=os.path.join(DATA_PATH, 'train.jsonl'), split='train')
val_dataset   = load_dataset('json',
    data_files=os.path.join(DATA_PATH, 'val.jsonl'),   split='train')

print(f'Train : {len(train_dataset):,}')
print(f'Val   : {len(val_dataset):,}')

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train : 8,000
Val   : 1,000


In [7]:
# Format dataset
def format_sample(s):
    return {'text': s['prompt'] + '\n' + s['completion'] + '</s>'}

train_dataset = train_dataset.map(format_sample)
val_dataset   = val_dataset.map(format_sample)
print('Formatted. Example:')
print(train_dataset[0]['text'][:300])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Formatted. Example:
You are a Python documentation expert. Write a clear, concise NumPy-style docstring for the following Python function.

### Function:
def debug(self, *args):
    if _canShortcutLogging(self.logCategory, DEBUG):
    return
    debugObject(self.logObjectName(), self.logCategory, *self.logFunction(*arg


In [8]:
#  Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print(f'Vocab size : {tokenizer.vocab_size:,}')

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Vocab size : 32,000


In [9]:
#  Load model in 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Model loaded in 4-bit.')

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded in 4-bit.


In [10]:
#  Attach LoRA adapters
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~0.5% trainable

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [11]:
#  Patch AdamW + Training arguments
import torch.optim as optim
import bitsandbytes as bnb

# Patch missing .train() method on all optimizer variants
for cls in [optim.AdamW,
            bnb.optim.AdamW, bnb.optim.AdamW8bit,
            bnb.optim.PagedAdamW, bnb.optim.PagedAdamW8bit]:
    if not hasattr(cls, 'train'):
        cls.train = lambda self: None

print('Patch applied.')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=False,
    optim='paged_adamw_8bit',
    report_to='none',
    push_to_hub=True,
    hub_model_id=REPO_NAME,
    hub_token=hf_token,
    hub_strategy='every_save',
)
print('Training args set.')

Patch applied.
Training args set.


In [12]:
# Train
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

print('Starting training...')
print(f'Total steps : {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS}')
print(f'Est. time   : ~3-4 hours on T4 x2')
print(f'Auto-pushing to HuggingFace every 200 steps.')
print(f'Safe to close laptop after you see step 200 in the progress bar.')
print()

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:413: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting training...
Total steps : 500
Est. time   : ~3-4 hours on T4 x2
Auto-pushing to HuggingFace every 200 steps.
Safe to close laptop after you see step 200 in the progress bar.



`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered intern

Step,Training Loss,Validation Loss
200,8.005500,1.011541
400,7.936300,0.994328


TrainOutput(global_step=500, training_loss=8.041043579101563, metrics={'train_runtime': 14142.6677, 'train_samples_per_second': 0.566, 'train_steps_per_second': 0.035, 'total_flos': 1.1188676894623334e+17, 'train_loss': 8.041043579101563, 'epoch': 1.0})

In [13]:
#  Final push after training completes
print('Pushing final model...')
trainer.push_to_hub()
tokenizer.push_to_hub(REPO_NAME, token=hf_token)

print()
print('=' * 60)
print(f'Model live at : https://huggingface.co/{REPO_NAME}')
print('=' * 60)
print()
print('NEXT STEPS:')
print('  1. Download this notebook -> save to train/train.ipynb in GitHub')
print('  2. Run eval.py on your laptop against data/test.jsonl')

Pushing final model...


README.md: 0.00B [00:00, ?B/s]


Model live at : https://huggingface.co/kk014/mistral-7b-docstring

NEXT STEPS:
  1. Download this notebook -> save to train/train.ipynb in GitHub
  2. Run eval.py on your laptop against data/test.jsonl


In [14]:
# Sanity check inference (run after Cell 13)
test_function = 'def calculate_bmi(weight_kg, height_m):\n    return weight_kg / (height_m ** 2)'

prompt = (
    'You are a Python documentation expert. '
    'Write a clear, concise NumPy-style docstring for the following Python function.\n\n'
    f'### Function:\n{test_function}\n\n'
    '### Docstring:'
)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('Generated docstring:')
print('-' * 40)
print(generated[len(prompt):].strip())

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Generated docstring:
----------------------------------------
Calculate the BMI of a person.

    Args:
        weight_kg (float): Weight in kilograms.
        height_m (float): Height in meters.

    Returns:
        float: BMI of the person.

    Raises:
        ValueError: If the height or weight is not a positive number.

    Example:
        >>> calculate_bmi(70, 1.75)
        21.970099999999998
        >>> calculate_bmi(70, 1.75)
        21.970099999999998
        >>> calculate_bmi(70, 1.75)
        21.970099999999998
        >>> calculate_b
